<div class="alert alert-block alert-info">
<b>ℹ️ Note on Execution Order</b><br>
This notebook expects the preprocessed data to already be saved. If you haven't yet, please run <code>02-preprocessing.ipynb</code> first so the required data is available to load.
</div>

## 1. Environment Setup

Install dependencies and initialize Ray + Spark (via RayDP) on SDSC Expanse (128 CPUs, 31 executors).

In [1]:
!pip install -qq pyspark ray raydp

import os
import sys
import logging
import socket
from pathlib import Path

os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"

import ray
import raydp

# Python-side loggers
logging.getLogger("ray").setLevel(logging.ERROR)
logging.getLogger("ray.data").setLevel(logging.ERROR)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import PCA
from pyspark.ml.feature import (
    RegexTokenizer,
    Word2Vec,
    VectorAssembler,
    StandardScaler,
    StopWordsRemover,
    Imputer,
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
# Clean notebook reruns
ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
    log_to_driver=False,
)

Python version:,3.11.6
Ray version:,2.54.0


In [3]:
spark = raydp.init_spark(
    app_name="FineWeb_Spark_on_Ray",
    num_executors=31,
    executor_cores=1,
    executor_memory="4GB",
)

In [4]:
print("=== Ray Cluster Resources ===")
for k, v in ray.cluster_resources().items():
    print(f"{k:25} : {v}")

print("\n=== Ray Available Resources ===")
for k, v in ray.available_resources().items():
    print(f"{k:25} : {v}")

print("\n=== Ray Worker Distribution Test ===")
@ray.remote
def where_am_i(i):
    return {
        "task": i,
        "pid": os.getpid(),
        "host": socket.gethostname()
    }

results = ray.get([where_am_i.remote(i) for i in range(50)])

# Summary
hosts = {r["host"] for r in results}
pids = {r["pid"] for r in results}

print(f"Unique hosts: {hosts}")
print(f"Unique worker processes: {len(pids)}\n")

# Table header
print(f"{'Task':<6} {'PID':<10} {'Host'}")
print("-" * 30)

# Table rows
for r in results[:10]:
    print(f"{r['task']:<6} {r['pid']:<10} {r['host']}")

=== Ray Cluster Resources ===
node:__internal_head__    : 1.0
object_store_memory       : 69561050726.0
node:198.202.103.12       : 1.0
memory                    : 162309118362.0
CPU                       : 128.0

=== Ray Available Resources ===
node:__internal_head__    : 1.0
memory                    : 29165132186.0
CPU                       : 97.0
object_store_memory       : 69561050726.0
node:198.202.103.12       : 1.0

=== Ray Worker Distribution Test ===
Unique hosts: {'exp-5-03'}
Unique worker processes: 50

Task   PID        Host
------------------------------
0      414350     exp-5-03
1      414349     exp-5-03
2      414358     exp-5-03
3      414354     exp-5-03
4      414353     exp-5-03
5      414359     exp-5-03
6      414361     exp-5-03
7      414352     exp-5-03
8      414367     exp-5-03
9      414366     exp-5-03


## 2. Data Loading & Exploration

Load the preprocessed FineWeb-Edu Parquet file, keep `text`, `token_count`, and `label`, and draw a random **20% sample** (142,415 rows, ~1.19 GB).

In [5]:
from fineweb_spark.paths import INTERIM_DATA_DIR

# Load preprocessed data
df = spark.read.parquet(str(INTERIM_DATA_DIR / "fineweb_preprocessed.parquet")) \
          .select("text", "token_count", "label") \
          .sample(fraction=0.2, seed=42)

In [6]:
print("===== DATASET INFO =====")

# number of rows
row_count = df.count()
print(f"Rows: {row_count:,}")

# number of columns
col_count = len(df.columns)
print(f"Columns: {col_count}")

# schema
print("Schema:")
df.printSchema()

# Estimated dataset size:
sample_rows = 1000
sample_pdf = df.limit(sample_rows).toPandas()

row_size_bytes = sample_pdf.memory_usage(deep=True).sum() / sample_rows
estimated_size_gb = (row_size_bytes * row_count) / (1024**3)

print("===== DATASET SIZE =====")
print(f"Estimated dataset size: {estimated_size_gb:.2f} GB")

print("\n===== DATASET EXAMPLES =====")
df.show(10, truncate=100)

===== DATASET INFO =====
Rows: 142,415
Columns: 3
Schema:
root
 |-- text: string (nullable = true)
 |-- token_count: long (nullable = true)
 |-- label: integer (nullable = true)

===== DATASET SIZE =====
Estimated dataset size: 1.19 GB

===== DATASET EXAMPLES =====
+----------------------------------------------------------------------------------------------------+-----------+-----+
|                                                                                                text|token_count|label|
+----------------------------------------------------------------------------------------------------+-----------+-----+
|Gravitational Wave Astronomy Will Be The Next Great Scientific Frontier\nBy Ethan Siegel\nImagine...|        730|    4|
|Locked inside the littlest objects of the solar system—asteroids, comets, and meteorites—is a sec...|        448|    5|
|This article explains the history and theory of the Socratic method of teaching, which emphasizes...|       1782|    4|
|Most ev

## 3. Train / Validation / Test Split

Split 60 / 20 / 20 (`seed=42`): **85,524** train / **28,389** val / **28,489** test rows.

In [7]:
# # Split into Train / Validation / Test (60 / 20 / 20)
train_data, val_data, test_data = df.randomSplit([0.6, 0.2, 0.2], seed=42)

print(f"Train: {train_data.count()}  Val: {val_data.count()}  Test: {test_data.count()}")

Train: 85524  Val: 28398  Test: 28493


## 4. Feature Engineering Pipeline

Build and fit a Spark ML `Pipeline` on training data only (no leakage): tokenise → remove stop words → Word2Vec (10-D) → impute token count → scale → assemble into an **11-D** feature vector.

In [8]:
# Split raw text into lowercase word tokens, removing punctuation
tokenizer = RegexTokenizer(inputCol="text", outputCol="raw_words", pattern="\\W+", toLowercase=True)

# Remove common English stop words to reduce noise and vocabulary size.
remover = StopWordsRemover(inputCol="raw_words", outputCol="filtered_words")

# Map filtered words to 10-dimensional dense vectors. Words appearing fewer than 500 times are ignored.
word2vec = Word2Vec(vectorSize=10, minCount=500, inputCol="filtered_words", outputCol="text_features")

# Impute missing token_count values using the mean strategy, creating a new column token_count_imputed
imputer = Imputer(inputCols=["token_count"], outputCols=["token_count_imputed"], strategy="mean")

# Assemble the imputed token_count into a vector for scaling
vec_assembler_num = VectorAssembler(inputCols=["token_count_imputed"], outputCol="token_vec")
scaler = StandardScaler(inputCol="token_vec", outputCol="token_scaled", withStd=True, withMean=True)

# Combine text features and scaled token_count into a single feature vector for modeling
final_assembler = VectorAssembler(
    inputCols=["text_features", "token_scaled"], 
    outputCol="features"
)

In [9]:
feature_pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    word2vec,
    imputer,
    vec_assembler_num,
    scaler,
    final_assembler
])

In [10]:
%%time
print("Fitting preprocessing pipeline...")
feature_model = feature_pipeline.fit(train_data)
print("Preprocessing pipeline fit complete.")

Fitting preprocessing pipeline...
Preprocessing pipeline fit complete.
CPU times: user 436 ms, sys: 193 ms, total: 629 ms
Wall time: 9min 55s


In [11]:
# Transform all splits and cache prepared datasets
print("Transforming and caching datasets...")

train_prepared = feature_model.transform(train_data).cache()
val_prepared   = feature_model.transform(val_data).cache()
test_prepared  = feature_model.transform(test_data).cache()

# materialize cache
train_prepared.count()
val_prepared.count()
test_prepared.count()

print("Cached prepared split sizes:")
print(f"Train prepared: {train_prepared.count():,}")
print(f"Val prepared  : {val_prepared.count():,}")
print(f"Test prepared : {test_prepared.count():,}")

Transforming and caching datasets...
Cached prepared split sizes:
Train prepared: 85,524
Val prepared  : 28,389
Test prepared : 28,489


In [12]:
# inspect columns
train_prepared.select("features", "label").show(5, truncate=100)

+----------------------------------------------------------------------------------------------------+-----+
|                                                                                            features|label|
+----------------------------------------------------------------------------------------------------+-----+
|[0.2383009260883192,0.2863086017057324,0.07962755052759857,0.17572874747603745,0.2439251993722343...|    4|
|[-0.12722733226447458,0.2433029048309609,-0.06664000812069958,0.3098260005673858,-0.0239955934649...|    3|
|[-0.005958844334583456,0.2329086061276564,0.059022778603504626,0.2806027898641833,0.0707080705674...|    3|
|[0.1545741775383552,-0.10050059875680342,0.008358807627382316,0.18643925381171655,0.1416257441831...|    3|
|[0.2387480125785939,-0.10186832588455029,0.09204794462531371,0.15950519863891433,0.13603482603758...|    3|
+----------------------------------------------------------------------------------------------------+-----+
only showing top 5 

## 5. PCA — Dimensionality Reduction

Fit PCA (`k=5`) on the training set and apply to all splits. Compresses the 11-D feature vector to **5 principal components**, then used as input to the classifiers.

In [13]:
print("Fitting PCA...")

pca = PCA(k=5, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(train_prepared)

train_prepared = pca_model.transform(train_prepared)
val_prepared   = pca_model.transform(val_prepared)
test_prepared  = pca_model.transform(test_prepared)

print("PCA transformation complete.")

Fitting PCA...
PCA transformation complete.


In [14]:
train_prepared.select("pca_features", "label").show(5, truncate=100)

+----------------------------------------------------------------------------------------------------+-----+
|                                                                                        pca_features|label|
+----------------------------------------------------------------------------------------------------+-----+
|[-0.3959947667071434,0.06082397855602066,-0.5098793631049477,-0.023589941490323924,-0.03670224044...|    4|
|[-0.13401022451080227,-0.15329166356527718,-0.25032289106791905,-0.19685738260737823,-0.031280530...|    3|
|[-0.33729969891449957,-0.07876235782923623,-0.2928729622172641,-0.10020900076154919,-0.0554540401...|    3|
|[-0.42243115052631175,0.17660659455183775,3.975137508176146E-4,0.03489191989831661,0.095345040942...|    3|
|[0.31111478241714785,0.17634111327730162,-0.12977106831141577,-0.016935411415499382,0.21721268063...|    3|
+----------------------------------------------------------------------------------------------------+-----+
only showing top 5 

## 6. Model Training

Train two Random Forest classifiers on the 5-D PCA features:
- **Model 1**: `numTrees=20`, `maxDepth=6` (~5.5 s)
- **Model 2**: `numTrees=80`, `maxDepth=10` (~58.6 s)

In [15]:
# Define model 1
rf1 = RandomForestClassifier(
    featuresCol="pca_features",
    labelCol="label",
    numTrees=20,
    maxDepth=6
)

In [16]:
%%time
print("Training Model 1...")
model1 = rf1.fit(train_prepared)
print("Model 1 complete.")

Training Model 1...
Model 1 complete.
CPU times: user 10.6 ms, sys: 7.73 ms, total: 18.3 ms
Wall time: 5.53 s


In [17]:
# Define model 2
rf2 = RandomForestClassifier(
    featuresCol="pca_features",
    labelCol="label",
    numTrees=80,
    maxDepth=10
)

In [18]:
%%time
print("Training Model 2...")
model2 = rf2.fit(train_prepared)
print("Model 2 complete.")

Training Model 2...
Model 2 complete.
CPU times: user 14.9 ms, sys: 4.97 s, total: 4.99 s
Wall time: 58.6 s


## 7. Evaluation & Predictions Analysis

Evaluate both models on train/val/test using accuracy, compare results, and inspect correct/FP/FN predictions from the best model. Majority-class baseline ≈ **52%**.

In [19]:
# Evaluate both models
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Model 1 predictions
train_preds1 = model1.transform(train_prepared)
val_preds1   = model1.transform(val_prepared)
test_preds1  = model1.transform(test_prepared)

train_acc1 = evaluator.evaluate(train_preds1)
val_acc1   = evaluator.evaluate(val_preds1)
test_acc1  = evaluator.evaluate(test_preds1)

print("\nModel 1 — RF (numTrees=20, maxDepth=6)")
print(f" Train Accuracy: {train_acc1:.4f}")
print(f" Val Accuracy  : {val_acc1:.4f}")
print(f" Test Accuracy : {test_acc1:.4f}")


# Model 2 predictions
train_preds2 = model2.transform(train_prepared)
val_preds2   = model2.transform(val_prepared)
test_preds2  = model2.transform(test_prepared)

train_acc2 = evaluator.evaluate(train_preds2)
val_acc2   = evaluator.evaluate(val_preds2)
test_acc2  = evaluator.evaluate(test_preds2)

print("\nModel 2 — RF (numTrees=80, maxDepth=10)")
print(f" Train Accuracy: {train_acc2:.4f}")
print(f" Val Accuracy  : {val_acc2:.4f}")
print(f" Test Accuracy : {test_acc2:.4f}")


Model 1 — RF (numTrees=20, maxDepth=6)
 Train Accuracy: 0.6391
 Val Accuracy  : 0.6365
 Test Accuracy : 0.6319

Model 2 — RF (numTrees=80, maxDepth=10)
 Train Accuracy: 0.6707
 Val Accuracy  : 0.6493
 Test Accuracy : 0.6458


In [20]:
# Compare results
print("\n--- Comparison ---")
print(f"{'Model':<35} {'Train':>8} {'Val':>8} {'Test':>8}")
print(f"{'RF (numTrees=20, maxDepth=6)':<35} {train_acc1:>8.4f} {val_acc1:>8.4f} {test_acc1:>8.4f}")
print(f"{'RF (numTrees=80, maxDepth=10)':<35} {train_acc2:>8.4f} {val_acc2:>8.4f} {test_acc2:>8.4f}")


--- Comparison ---
Model                                  Train      Val     Test
RF (numTrees=20, maxDepth=6)          0.6391   0.6365   0.6319
RF (numTrees=80, maxDepth=10)         0.6707   0.6493   0.6458


In [21]:
# Pick best model
results = [
    ("RF (numTrees=20, maxDepth=6)", model1, test_acc1),
    ("RF (numTrees=80, maxDepth=10)", model2, test_acc2),
]

best_name, best_model, best_acc = max(results, key=lambda x: x[2])

print("\nBest model:")
print(f" Name          : {best_name}")
print(f" Test accuracy : {best_acc:.4f}")


Best model:
 Name          : RF (numTrees=80, maxDepth=10)
 Test accuracy : 0.6458


## 8. Save & Cleanup

Save the best model to disk, unpersist cached DataFrames, and shut down Ray.

In [22]:
# Save model
from fineweb_spark.paths import MODELS_DIR

best_model.write().overwrite().save(str(MODELS_DIR / "best_model2-pca"))

print(f"Model saved")

Model saved


In [23]:
# Cleanup cache and shutdown the ray
train_prepared.unpersist()
val_prepared.unpersist()
test_prepared.unpersist()

ray.shutdown()
print("Done.")


Done.


## Fitting Analysis

**1. Where does the model sit on the fitting graph?**

Dataset: 142,415 rows, binary classification (`label` 3 vs. 4), baseline ≈ 52%.

| Model | Train | Val | Test | Gap |
|---|---|---|---|---|
| RF (20 trees, depth 6) | 0.6391 | 0.6365 | 0.6319 | 0.0072 |
| RF (80 trees, depth 10) | 0.6707 | 0.6493 | 0.6458 | 0.0249 |

Model 1 sits near **mild underfitting** (small gap, limited capacity). Model 2 shows **mild overfitting** (2.49 pp gap) but is well-controlled and delivers the best accuracy.

**2. How does PCA affect results vs. the full feature set?**

| Approach | Test Accuracy |
|---|---|
| No PCA (Model 1 notebook, RF depth 8) | **68.66%** |
| PCA k=5 (this notebook, RF depth 10) | **64.58%** |

Reducing 11 → 5 dimensions costs ~4.1 pp accuracy but cuts training time from ~13 min to under 1 min.

**3. Potential improvements**

- Increase `k` to 8–10 to retain more variance before the elbow
- Add richer features (TF-IDF, sentence count) to the input before PCA
- Try `GBTClassifier` on PCA features for better handling of the binary boundary
- Include `label=5` to restore the full 3-class problem

## Conclusion

### Model 1 — No PCA (`03-model-1.ipynb`)

Trained on the full 11-D feature vector. Best configuration: RF (`numTrees=20`, `maxDepth=8`) → **68.66% test accuracy**, 1.43 pp train–test gap. Good generalisation; the pipeline signal (Word2Vec + token count) is confirmed.

### Model 2 — PCA k=5 (this notebook)

Trained on 5 PCA components. Best configuration: RF (`numTrees=80`, `maxDepth=10`) → **64.58% test accuracy**, 2.49 pp gap. PCA cuts training time dramatically but loses ~4 pp of accuracy due to aggressive compression.

**Key finding**: PCA (`k=5`) is too aggressive here — the lost variance carries real discriminative signal for the Random Forest.

**What can be done to improve it?**

- Use a larger `k` (8–10) or inspect the explained-variance curve to pick the elbow
- Enrich the input features before PCA (TF-IDF, stylistic features) to make compression less lossy
- Try `GBTClassifier` which may extract more signal from the reduced representation
- Include `label=5` and train a proper 3-class model
- Scale to a larger data fraction to reduce variance across all metrics